# Dasai Mochi — OLED Drawing & Animation Lab

Welcome! In this notebook, we will build an expressive animated character on a small monochrome OLED, starting from the absolute basics.

Before animation, learn to control the screen: **write text, plot pixels, draw lines and shapes, fill shapes, rotate the display, and erase/redraw content**. The OLED is a pixel canvas; animation is made by changing what is drawn over time.

## What we will use
- A 128 × 64 SSD1306 monochrome OLED (I²C, commonly address `0x3C`), connected to a NodeMCU ESP8266 (ESP-12)
- NodeMCU ESP8266 (ESP-12), programmed with the ESP8266 Arduino core; examples use `Adafruit_GFX` and `Adafruit_SSD1306`
- For the NodeMCU ESP8266 (ESP-12), use SDA `D2` (GPIO4) and SCL `D1` (GPIO5) in this example; verify your module's pinout.

The code cells below are **Arduino/C++ sketches**, not ordinary Python notebook cells. Copy each stage into your Arduino workflow / Embedded Notebook and verify it for your connected board.

## Basic drawing vocabulary
- **Text:** `setCursor(x, y)`, `setTextSize(n)`, `print()` / `println()`
- **Line:** `drawLine(x1, y1, x2, y2, color)`
- **Outline shapes:** `drawRect()`, `drawCircle()`, `drawRoundRect()`, `drawTriangle()`
- **Filled shapes:** `fillRect()`, `fillCircle()`, `fillRoundRect()`, `fillTriangle()`
- **Pixel:** `drawPixel(x, y, color)`
- **Show changes:** `display.display()` sends the drawing buffer to the OLED
- **Erase:** `clearDisplay()` clears the whole buffer; then call `display.display()` to show the cleared screen

Coordinates start at the top-left: `(0,0)`. x increases rightward and y increases downward. For this 128 × 64 display, valid coordinates are x = 0–127 and y = 0–63.

## Rotation and erasing
`setRotation(0)`, `setRotation(1)`, `setRotation(2)`, and `setRotation(3)` rotate the display coordinate orientation in 90° steps. This is **not arbitrary-angle rotation of a drawn object**. For arbitrary rotation, calculate transformed point coordinates mathematically and redraw the shape.

Most simple OLED animation erases by clearing the frame and drawing it again. To make an object disappear, either clear the whole display and redraw everything, or redraw the old object area in the background colour before drawing the new frame. The first approach is easiest for Mochi.


# 00 — OLED drawing basics

This first sketch demonstrates text, pixels, lines, outline and filled shapes, screen rotation, and clearing the display. Try changing the coordinates and dimensions to see how the 128 × 64 pixel canvas behaves.

**Important:** drawing functions update a RAM buffer. `display.display()` transfers that buffer to the physical OLED. `clearDisplay()` clears the buffer; it does not visibly clear the panel until `display.display()` is called.


In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

#define SCREEN_WIDTH 128
#define SCREEN_HEIGHT 64
#define OLED_RESET -1
#define OLED_ADDR 0x3C

Adafruit_SSD1306 display(SCREEN_WIDTH, SCREEN_HEIGHT, &Wire, OLED_RESET);

void setup() {
  Wire.begin(D2, D1);  // NodeMCU ESP8266: SDA=D2 (GPIO4), SCL=D1 (GPIO5)
  if (!display.begin(SSD1306_SWITCHCAPVCC, OLED_ADDR)) {
    while (true) {}    // Stop if the OLED is not detected
  }

  display.setRotation(0);  // Try 0, 1, 2, or 3 (90-degree steps)
  display.clearDisplay();

  // 1) Write text
  display.setTextColor(SSD1306_WHITE);
  display.setTextSize(1);
  display.setCursor(2, 2);
  display.println("OLED drawing basics");

  // 2) Draw an individual pixel and a line
  display.drawPixel(4, 18, SSD1306_WHITE);
  display.drawLine(8, 18, 55, 18, SSD1306_WHITE);

  // 3) Outline shapes
  display.drawRect(4, 25, 24, 15, SSD1306_WHITE);
  display.drawCircle(43, 32, 8, SSD1306_WHITE);
  display.drawRoundRect(58, 25, 26, 15, 4, SSD1306_WHITE);
  display.drawTriangle(91, 40, 103, 23, 116, 40, SSD1306_WHITE);

  // 4) Filled shapes
  display.fillRect(4, 46, 18, 12, SSD1306_WHITE);
  display.fillCircle(37, 52, 6, SSD1306_WHITE);
  display.fillRoundRect(52, 46, 25, 12, 4, SSD1306_WHITE);
  display.fillTriangle(91, 58, 103, 44, 116, 58, SSD1306_WHITE);

  // 5) Send the completed frame to the OLED
  display.display();
}

void loop() {
  // 6) Erase everything after a pause, then show the blank screen
  delay(2000);
  display.clearDisplay();
  display.display();

  // Re-draw content here if you want it to remain visible or animate.
  while (true) {}
}


### Quick experiments
- Change `setTextSize(1)` to `2` and observe how text occupies more pixels.
- Change `drawLine()` endpoints to change line angle and length.
- Replace `drawRect()` with `fillRect()` to compare outline and solid shapes.
- Change `setRotation(0)` to `1`, `2`, or `3`. The logical width and height are swapped for rotations 1 and 3; take care that your drawing coordinates fit the rotated canvas.
- `SSD1306_BLACK` draws in the background colour and can erase a small shape in the buffer. For example, `fillCircle(oldX, oldY, radius, SSD1306_BLACK)` can erase a circle before drawing it at a new position. For complex scenes, clear and redraw the full frame to avoid leftover pixels.
- This library supports 90° display rotation, not a direct `rotate(37°)` call. For arbitrary-angle object rotation, transform each point around a pivot using sine and cosine, then redraw the transformed points.

**Transition to animation:** once these static drawing commands make sense, the following stages reuse them to create eyes, expressions, and motion.


# OLED pixel matrix — the real coordinate system behind Mochi

This notebook targets a **128 × 64 pixel SSD1306 OLED**, a grid of 8,192 individually addressable pixels.

```text
                  128 pixels wide
        x=0 ┌──────────────────────────────────────────────┐ x=127
            │                                              │
            │                 OLED                         │
            │                128 × 64                      │ 64
            │                                              │ pixels
            │                                              │ high
        y=63└──────────────────────────────────────────────┘
             y=0 is the top; y increases downward
```

- Valid x-coordinates: `0 … 127`; valid y-coordinates: `0 … 63`
- Total pixels: `128 × 64 = 8,192`
- Matrix aspect ratio: `128:64 = 2:1` (twice as wide as it is tall).

## Shape sizes relative to the matrix

| Element | Pixels | Share of display |
|---|---:|---:|
| Display width | 128 px | 100% width |
| Display height | 64 px | 100% height |
| Typical eye width | 28 px | 21.9% width |
| Typical eye height | 40 px | 62.5% height |
| Eye centre gap (x=36 to x=92) | 56 px | 43.8% width |
| Left eye centre (x=36) | 36 px | 28.1% from left |
| Right eye centre (x=92) | 92 px | 71.9% from left |

A `28 × 40 px` eye has a width-to-height ratio of `0.7:1`, so it is taller than it is wide. The screen's `2:1` ratio leaves room for two eyes side by side.

## Pixel coordinates versus physical size

Coordinates are pixels, not millimetres. If the active display area is `W_mm × H_mm`, approximate pixel pitch is `W_mm / 128` horizontally and `H_mm / 64` vertically. Physical dimensions depend on the OLED module; consult its datasheet for enclosure design.

## How animation uses the matrix

- **Translation:** change an eye centre `(cx, cy)` in pixels.
- **Scaling:** change eye width and height in pixels.
- **Blinking:** reduce eye height toward a few pixels, then restore it.
- **Sine-wave bounce:** add a time-varying pixel offset to `cy`.
- **Clipping:** keep drawing within `x=0…127`, `y=0…63`.

Motion may be calculated with floating-point values, but the OLED lights whole pixels. Tiny changes can round to the same pixel for a frame and then visibly jump by one pixel. Also, a 4 px horizontal offset is 3.125% of width, while a 4 px vertical offset is 6.25% of height. Choose x/y amplitudes relative to their own axes.


# 01 — One static eye

Start with the smallest useful visual primitive.

A Mochi eye is a rounded rectangle:

```text
   ████████
  ██████████
  ██████████
   ████████
```

There is no animation yet. This cell teaches the OLED coordinate system and basic geometry.

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128, 64, &Wire, -1);

void setup() {
  Wire.begin(21, 22);
  display.begin(SSD1306_SWITCHCAPVCC, 0x3C);

  display.clearDisplay();

  // x=50, y=12, width=28, height=40.
  display.fillRoundRect(50, 12, 28, 40, 8, WHITE);

  display.display();
}

void loop() {}

# 02 — Two eyes

The character is created by **composition**: draw the same primitive twice.

```text
    ██████       ██████
   ████████     ████████
   ████████     ████████
```

Keeping the dimensions identical gives a symmetric face.

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128, 64, &Wire, -1);

void setup() {
  Wire.begin(21, 22);
  display.begin(SSD1306_SWITCHCAPVCC, 0x3C);

  display.clearDisplay();

  display.fillRoundRect(22, 12, 28, 40, 8, WHITE);
  display.fillRoundRect(78, 12, 28, 40, 8, WHITE);

  display.display();
}

void loop() {}

# 03 — Reusable `drawEye()`

Hard-coded drawing does not scale.

Create one function:

```cpp
drawEye(centerX, centerY, width, height);
```

Now the same function can later draw:

- normal eyes
- smaller SAD eyes
- larger SURPRISED eyes
- partially closed eyes
- different expressions

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128, 64, &Wire, -1);

void drawEye(int cx, int cy, int w, int h) {
  display.fillRoundRect(
    cx - w / 2,
    cy - h / 2,
    w, h,
    min(10, h / 2),
    WHITE
  );
}

void setup() {
  Wire.begin(21, 22);
  display.begin(SSD1306_SWITCHCAPVCC, 0x3C);

  display.clearDisplay();

  drawEye(36, 32, 28, 40);
  drawEye(92, 32, 28, 40);

  display.display();
}

void loop() {}

# 04 — Turn time into frames

Use `millis()` instead of blocking `delay()` for the animation clock.

```cpp
if (now - lastFrame >= 33) {
    lastFrame = now;
    // generate next frame
}
```

33 ms gives roughly:

\[
1000 / 33 \approx 30 FPS
\]

This is the frame-rate mechanism used by the final Mochi.

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128, 64, &Wire, -1);
unsigned long lastFrame = 0;

void drawEye(int cx, int cy, int w, int h) {
  display.fillRoundRect(cx-w/2, cy-h/2, w, h, min(10,h/2), WHITE);
}

void setup() {
  Wire.begin(21, 22);
  display.begin(SSD1306_SWITCHCAPVCC, 0x3C);
}

void loop() {
  unsigned long now = millis();

  if (now - lastFrame < 33) return;
  lastFrame = now;

  display.clearDisplay();
  drawEye(36, 32, 28, 40);
  drawEye(92, 32, 28, 40);
  display.display();
}

# 05 — Direct movement

A position is just a variable.

```cpp
eyeX += 0.5;
```

Every new frame uses the new value, so the eye moves.

This is the simplest form of procedural animation:

> **Change a parameter → redraw → repeat.**

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128, 64, &Wire, -1);
unsigned long lastFrame = 0;
float eyeX = 0;

void drawEye(int cx, int cy, int w, int h) {
  display.fillRoundRect(cx-w/2, cy-h/2, w, h, min(10,h/2), WHITE);
}

void setup() {
  Wire.begin(21, 22);
  display.begin(SSD1306_SWITCHCAPVCC, 0x3C);
}

void loop() {
  unsigned long now = millis();
  if (now - lastFrame < 33) return;
  lastFrame = now;

  eyeX += 0.5;

  if (eyeX > 10 || eyeX < -10)
    eyeX *= -1;

  display.clearDisplay();
  drawEye(36 + eyeX, 32, 28, 40);
  drawEye(92 + eyeX, 32, 28, 40);
  display.display();
}

# 06 — Current position → target position

Now introduce the most important movement algorithm:

```cpp
current += (target - current) * speed;
```

This is **exponential smoothing**.

If:

```text
current = 0
target  = 10
speed   = 0.3
```

then:

```text
0
 ↓
3
 ↓
5.1
 ↓
6.57
 ↓
7.60
 ↓
...
10
```

The object moves quickly at first and slows as it approaches the target.

This same equation is later reused for:

- eye position
- eye openness
- eye size
- smooth transitions

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128, 64, &Wire, -1);

unsigned long lastFrame = 0;
float currentX = 0;
float targetX = 10;

void drawEye(int cx, int cy, int w, int h) {
  display.fillRoundRect(cx-w/2, cy-h/2, w, h, min(10,h/2), WHITE);
}

void setup() {
  Wire.begin(21, 22);
  display.begin(SSD1306_SWITCHCAPVCC, 0x3C);
}

void loop() {
  unsigned long now = millis();
  if (now - lastFrame < 33) return;
  lastFrame = now;

  currentX += (targetX - currentX) * 0.3;

  if (abs(currentX - targetX) < 0.5)
    targetX = -targetX;

  display.clearDisplay();
  drawEye(36 + currentX, 32, 28, 40);
  drawEye(92 + currentX, 32, 28, 40);
  display.display();
}

# 07 — Blinking is also target-following

Blinking uses the exact same idea.

```text
open:
    targetOpen = 1

blink:
    targetOpen = 0
```

Then:

```cpp
openness += (targetOpen - openness) * 0.5;
```

So blinking is not a completely different animation system.

It is simply **interpolation applied to eye height**.

A timer decides *when* the target changes.

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128, 64, &Wire, -1);

unsigned long lastFrame = 0;
unsigned long nextBlink = 0;
bool blinking = false;

float openness = 1.0;
float targetOpen = 1.0;

void drawEye(int cx, int cy, int w, int h) {
  h = max(3, h);
  display.fillRoundRect(cx-w/2, cy-h/2, w, h, min(8,h/2), WHITE);
}

void setup() {
  Wire.begin(21, 22);
  display.begin(SSD1306_SWITCHCAPVCC, 0x3C);

  randomSeed(esp_random());
  nextBlink = millis() + 2000;
}

void loop() {
  unsigned long now = millis();
  if (now - lastFrame < 33) return;
  lastFrame = now;

  if (!blinking && now >= nextBlink) {
    blinking = true;
    targetOpen = 0;
  }

  if (blinking && now >= nextBlink + 130) {
    blinking = false;
    targetOpen = 1;
    nextBlink = now + random(1500, 4500);
  }

  openness += (targetOpen - openness) * 0.5;

  display.clearDisplay();

  int h = max(3, (int)(40 * openness));

  drawEye(36, 32, 28, h);
  drawEye(92, 32, 28, h);

  display.display();
}

# 08 — Random looking

Now combine:

```text
randomness → chooses target
interpolation → moves toward target
```

Every 1–2.5 seconds:

```cpp
targetX = random(-10, 11);
targetY = random(-6, 7);
```

The eye then smoothly approaches the new location.

This is why Mochi can look alive without storing any animation frames.

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128, 64, &Wire, -1);

unsigned long lastFrame = 0;
unsigned long nextLook = 0;

float curX = 0, curY = 0;
float tgtX = 0, tgtY = 0;

void drawEye(int cx, int cy, int w, int h) {
  display.fillRoundRect(cx-w/2, cy-h/2, w, h, min(8,h/2), WHITE);
}

void setup() {
  Wire.begin(21, 22);
  display.begin(SSD1306_SWITCHCAPVCC, 0x3C);

  randomSeed(esp_random());
  nextLook = millis();
}

void loop() {
  unsigned long now = millis();
  if (now - lastFrame < 33) return;
  lastFrame = now;

  if (now >= nextLook) {
    tgtX = random(-10, 11);
    tgtY = random(-6, 7);
    nextLook = now + random(1000, 2500);
  }

  curX += (tgtX - curX) * 0.3;
  curY += (tgtY - curY) * 0.3;

  display.clearDisplay();

  drawEye(36 + curX, 32 + curY, 28, 40);
  drawEye(92 + curX, 32 + curY, 28, 40);

  display.display();
}

# 09 — Sine waves: bounce, breathe and pulse

Periodic motion can be generated by:

\[
y=A\sin(\omega t)
\]

For HAPPY:

```cpp
bounce = sin(time * 0.012) * 3;
```

A sine wave naturally gives:

```text
        ╭──╮
      ╭─╯  ╰─╮
──────╯      ╰──────
```

Later the same idea becomes:

- HAPPY → bounce
- SLEEPY → breathing
- LOVE → heartbeat

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128, 64, &Wire, -1);
unsigned long lastFrame = 0;

void drawEye(int cx, int cy, int w, int h) {
  display.fillRoundRect(cx-w/2, cy-h/2, w, h, min(8,h/2), WHITE);
}

void setup() {
  Wire.begin(21, 22);
  display.begin(SSD1306_SWITCHCAPVCC, 0x3C);
}

void loop() {
  unsigned long now = millis();
  if (now - lastFrame < 33) return;
  lastFrame = now;

  int bounce = (int)(sin(now * 0.012) * 3);

  display.clearDisplay();
  drawEye(36, 32 + bounce, 28, 40);
  drawEye(92, 32 + bounce, 28, 40);
  display.display();
}

# 10 — Emotion-specific geometry

The normal eye is a rounded rectangle.

ANGRY modifies it with a black triangle:

```text
████████████
██████████
██████
██
```

SAD/TIRED use a different mask to create drooping eyelids.

This is a useful microcontroller graphics technique:

```text
base shape
    +
mask
    ↓
new expression
```

No bitmap animation frames are required.

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128, 64, &Wire, -1);

enum Mood { NORMAL, ANGRY, SAD };
Mood mood = ANGRY;

void drawEye(int cx, int cy, int w, int h, bool leftEye) {
  int x = cx - w/2;
  int y = cy - h/2;

  display.fillRoundRect(x, y, w, h, 8, WHITE);

  if (mood == ANGRY) {
    if (leftEye)
      display.fillTriangle(x-1,y-1,x+w+1,y-1,x+w+1,y+h*4/10,BLACK);
    else
      display.fillTriangle(x-1,y-1,x+w+1,y-1,x-1,y+h*4/10,BLACK);
  }

  if (mood == SAD) {
    int drop = h * 5 / 10;

    if (leftEye)
      display.fillTriangle(x-1,y-1,x+w+1,y-1,x-1,y+drop,BLACK);
    else
      display.fillTriangle(x-1,y-1,x+w+1,y-1,x+w+1,y+drop,BLACK);
  }
}

void setup() {
  Wire.begin(21,22);
  display.begin(SSD1306_SWITCHCAPVCC,0x3C);

  display.clearDisplay();
  drawEye(36,32,32,40,true);
  drawEye(92,32,32,40,false);
  display.display();
}

void loop() {}

# 11 — HAPPY, LOVE and SURPRISED shapes

Three different strategies:

### HAPPY

Several circles form an arch, then a black rectangle masks the lower half:

```text
   ╭────╮
  ╱      ╲
```

### LOVE

A heart is constructed from:

```text
circle + circle + triangle
```

### SURPRISED

The normal rounded rectangle becomes larger and gets a circular pupil.

This demonstrates that emotion can modify:

- shape
- size
- extra geometry
- motion

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128,64,&Wire,-1);

enum Mood { HAPPY, LOVE, SURPRISED };
Mood mood = LOVE;

void drawHeart(int cx,int cy,int s) {
  display.fillCircle(cx-s/2,cy-s/4,s/2+1,WHITE);
  display.fillCircle(cx+s/2,cy-s/4,s/2+1,WHITE);
  display.fillTriangle(cx-s-1,cy-s/6,cx+s+1,cy-s/6,cx,cy+s,WHITE);
}

void drawEye(int cx,int cy) {
  if (mood == HAPPY) {
    for (int i=0;i<4;i++)
      display.drawCircle(cx,cy+8,13+i,WHITE);

    display.fillRect(cx-20,cy+8,41,30,BLACK);
    return;
  }

  if (mood == LOVE) {
    int s = 10 + (int)(3*sin(millis()*0.01));
    drawHeart(cx,cy,s);
    return;
  }

  int w=36, h=50;

  display.fillRoundRect(cx-w/2,cy-h/2,w,h,10,WHITE);
  display.fillCircle(cx,cy,6,BLACK);
}

void setup() {
  Wire.begin(21,22);
  display.begin(SSD1306_SWITCHCAPVCC,0x3C);
}

void loop() {
  display.clearDisplay();
  drawEye(36,32);
  drawEye(92,32);
  display.display();
  delay(33);
}

# 12 — SAD tears with phase

A tear can be animated using a repeating phase:

```cpp
phase = (time / 25 + offset) % 100;
```

Then map phase to vertical position:

```cpp
tearY = eyeBottom + phase * 16 / 100;
```

The result is:

```text
  👁
  ●
  |
  ●
  |
  |
  ●
```

The two tears use different offsets:

```text
left  → +0
right → +50
```

so they do not fall in perfect synchronization.

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128,64,&Wire,-1);

void drawSadEye(int cx,int cy,bool leftEye) {
  int w=30, h=32;
  int x=cx-w/2, y=cy-h/2;

  display.fillRoundRect(x,y,w,h,8,WHITE);

  int drop=h/2;

  if(leftEye)
    display.fillTriangle(x-1,y-1,x+w+1,y-1,x-1,y+drop,BLACK);
  else
    display.fillTriangle(x-1,y-1,x+w+1,y-1,x+w+1,y+drop,BLACK);

  int phase=(millis()/25+(leftEye?0:50))%100;
  int tearY=y+h+phase*16/100;

  if(phase<90 && tearY<62)
    display.fillCircle(cx,tearY,2,WHITE);
}

void setup() {
  Wire.begin(21,22);
  display.begin(SSD1306_SWITCHCAPVCC,0x3C);
}

void loop() {
  display.clearDisplay();

  drawSadEye(36,32,true);
  drawSadEye(92,32,false);

  display.display();
  delay(25);
}

# 13 — SLEEPY Z's with phase offsets

Three Z's use one animation equation:

```cpp
phase = (time / 40 + i * 33) % 100;
```

The `i * 33` term gives each Z a different phase.

```text
Z1 → phase + 0
Z2 → phase + 33
Z3 → phase + 66
```

This is a general technique for repeated effects such as particles, LEDs and waves.

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128,64,&Wire,-1);

void drawSleepZ() {
  display.setTextColor(WHITE);

  for(int i=0;i<3;i++) {
    int phase=(millis()/40+i*33)%100;

    int x=92+phase/6+i*4;
    int y=28-phase*28/100;

    display.setTextSize(i==2?2:1);
    display.setCursor(x,y);
    display.print(i==2?"Z":"z");
  }
}

void setup() {
  Wire.begin(21,22);
  display.begin(SSD1306_SWITCHCAPVCC,0x3C);
}

void loop() {
  display.clearDisplay();

  display.fillRoundRect(22,38,28,5,2,WHITE);
  display.fillRoundRect(78,38,28,5,2,WHITE);

  drawSleepZ();

  display.display();
  delay(33);
}

# 14 — Mood as a finite-state machine

Now the character gets a real emotional state:

```cpp
enum Mood {
    NORMAL,
    HAPPY,
    ANGRY,
    TIRED,
    SLEEPY,
    LOVE,
    SAD,
    SURPRISED
};
```

The state determines:

```text
shape
size
looking direction
blink behaviour
special effects
motion
```

The important architecture becomes:

```text
MOOD
  ↓
BEHAVIOUR
  ↓
ANIMATION VARIABLES
  ↓
DRAWING
```

Automatic cycling is simply:

```cpp
mood = (Mood)((mood + 1) % MOOD_COUNT);
```

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128,64,&Wire,-1);

enum Mood {
  NORMAL,HAPPY,ANGRY,TIRED,
  SLEEPY,LOVE,SAD,SURPRISED,
  MOOD_COUNT
};

Mood mood=NORMAL;

unsigned long nextMood=0;
const unsigned long MOOD_TIME=2000;

void drawCharacter() {
  display.clearDisplay();

  switch(mood) {
    case NORMAL:
      display.fillRoundRect(22,12,28,40,8,WHITE);
      display.fillRoundRect(78,12,28,40,8,WHITE);
      break;

    case HAPPY:
      display.drawCircle(36,40,14,WHITE);
      display.drawCircle(92,40,14,WHITE);
      break;

    case ANGRY:
      display.fillRoundRect(22,12,28,40,8,WHITE);
      display.fillRoundRect(78,12,28,40,8,WHITE);
      display.fillTriangle(21,11,51,11,51,27,BLACK);
      display.fillTriangle(77,11,107,11,77,27,BLACK);
      break;

    case SLEEPY:
      display.fillRoundRect(22,38,28,5,2,WHITE);
      display.fillRoundRect(78,38,28,5,2,WHITE);
      break;

    default:
      display.fillRoundRect(22,12,28,40,8,WHITE);
      display.fillRoundRect(78,12,28,40,8,WHITE);
      break;
  }

  display.display();
}

void setup() {
  Wire.begin(21,22);
  display.begin(SSD1306_SWITCHCAPVCC,0x3C);
  nextMood=millis()+MOOD_TIME;
}

void loop() {
  unsigned long now=millis();

  if(now>=nextMood) {
    mood=(Mood)((mood+1)%MOOD_COUNT);
    nextMood=now+MOOD_TIME;
  }

  drawCharacter();
  delay(33);
}

# 15 — Serial interaction

Now the user can control the emotional state.

```text
n → NORMAL
h → HAPPY
a → ANGRY
t → TIRED
s → SLEEPY
l → LOVE
d → SAD
u → SURPRISED
```

Notice the design principle:

> Input changes the **state**. It does not directly draw pixels.

That separation makes the system much easier to extend.

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128,64,&Wire,-1);

enum Mood {
  NORMAL,HAPPY,ANGRY,TIRED,
  SLEEPY,LOVE,SAD,SURPRISED
};

Mood mood=NORMAL;

void setMood(Mood newMood) {
  mood=newMood;
}

void handleSerial() {
  while(Serial.available()) {
    char c=Serial.read();

    if(c=='n') setMood(NORMAL);
    else if(c=='h') setMood(HAPPY);
    else if(c=='a') setMood(ANGRY);
    else if(c=='t') setMood(TIRED);
    else if(c=='s') setMood(SLEEPY);
    else if(c=='l') setMood(LOVE);
    else if(c=='d') setMood(SAD);
    else if(c=='u') setMood(SURPRISED);
  }
}

void drawCharacter() {
  display.clearDisplay();

  if(mood==HAPPY) {
    display.drawCircle(36,40,14,WHITE);
    display.drawCircle(92,40,14,WHITE);
  }
  else if(mood==LOVE) {
    display.fillCircle(31,27,8,WHITE);
    display.fillCircle(41,27,8,WHITE);
    display.fillTriangle(23,29,49,29,36,52,WHITE);

    display.fillCircle(87,27,8,WHITE);
    display.fillCircle(97,27,8,WHITE);
    display.fillTriangle(79,29,105,29,92,52,WHITE);
  }
  else {
    display.fillRoundRect(22,12,28,40,8,WHITE);
    display.fillRoundRect(78,12,28,40,8,WHITE);
  }

  display.display();
}

void setup() {
  Serial.begin(115200);
  Wire.begin(21,22);
  display.begin(SSD1306_SWITCHCAPVCC,0x3C);

  Serial.println("n/h/a/t/s/l/d/u");
}

void loop() {
  handleSerial();
  drawCharacter();
  delay(33);
}

# 16 — Touch interaction

The same state-machine idea can be connected to hardware input.

For an ESP32 touch pad:

```cpp
touchRead(TOUCH_PIN)
```

can be used as the trigger.

For example:

```text
touch
  ↓
LOVE ❤️
  ↓
hold LOVE for 3 seconds
  ↓
automatic behaviour resumes
```

This is the point where the animation becomes an **interactive character** instead of only a display.

In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

Adafruit_SSD1306 display(128,64,&Wire,-1);

#define TOUCH_PIN 4
#define TOUCH_THRESHOLD 30

enum Mood { NORMAL, LOVE };
Mood mood=NORMAL;

unsigned long loveUntil=0;

void setup() {
  Wire.begin(21,22);
  display.begin(SSD1306_SWITCHCAPVCC,0x3C);
}

void loop() {
  unsigned long now=millis();

  if(touchRead(TOUCH_PIN)<TOUCH_THRESHOLD) {
    mood=LOVE;
    loveUntil=now+3000;
  }

  if(now>=loveUntil && mood==LOVE) {
    mood=NORMAL;
  }

  display.clearDisplay();

  if(mood==LOVE) {
    display.fillCircle(31,27,8,WHITE);
    display.fillCircle(41,27,8,WHITE);
    display.fillTriangle(23,29,49,29,36,52,WHITE);

    display.fillCircle(87,27,8,WHITE);
    display.fillCircle(97,27,8,WHITE);
    display.fillTriangle(79,29,105,29,92,52,WHITE);
  }
  else {
    display.fillRoundRect(22,12,28,40,8,WHITE);
    display.fillRoundRect(78,12,28,40,8,WHITE);
  }

  display.display();
  delay(33);
}

# 17 — Final modular Mochi

This cell is the **complete structured project implementation**.

The final architecture is:

```text
                 USER / TIMER
                      │
                      ▼
              updateBehavior()
                      │
          ┌───────────┼───────────┐
          ↓           ↓           ↓
       mood        blink       targets
          │           │           │
          └───────────┼───────────┘
                      ↓
                  animate()
                      │
          ┌───────────┼───────────┐
          ↓           ↓           ↓
       position    openness      size
          │           │           │
          └───────────┼───────────┘
                      ↓
                  drawEyes()
                      │
          ┌───────────┼───────────┐
          ↓           ↓           ↓
       geometry     effects      text
          │           │           │
          └───────────┼───────────┘
                      ↓
                    OLED
```

The final source below is the structured project version, included as the reference implementation after all the learning cells.

In [ ]:
/*
 * ==========================================================================
 * Dasai Mochi - Animated Desk Pet
 * ==========================================================================
 * Board  : ESP32 Dev Module
 * Display: SSD1306 128x64 I2C OLED
 * Libraries:
 *   - Adafruit GFX Library
 *   - Adafruit SSD1306
 *
 * Wiring
 * -------
 * OLED VCC -> 3V3
 * OLED GND -> GND
 * OLED SDA -> GPIO21
 * OLED SCL -> GPIO22
 *
 * Optional touch input:
 *   Touch pad -> GPIO4
 *   Enable USE_TOUCH below.
 *
 * Moods and their animations
 * --------------------------
 * NORMAL    : random looking + blinking
 * HAPPY     : ^ ^ eyes + bouncing
 * ANGRY     : slanted eyelids + shaking
 * TIRED     : droopy eyelids + slow/heavy blinking
 * SLEEPY    : almost closed eyes + breathing + floating Z's
 * LOVE      : pulsing heart eyes
 * SAD       : small downward-looking eyes + falling tears
 * SURPRISED : large eyes + small moving pupils
 *
 * Animation architecture
 * -----------------------
 * The character is NOT made from pre-rendered animation frames.
 * Each frame is generated mathematically:
 *
 *   1. updateBehavior() chooses targets and handles timed events.
 *   2. animate() smoothly moves the current values toward those targets.
 *   3. drawEyes() converts those values into pixels and sends the frame
 *      to the OLED.
 *
 * Main animation techniques used:
 *   - Target interpolation / exponential smoothing
 *   - Sine waves for periodic motion
 *   - Randomness for natural-looking behavior
 *   - Phase offsets for repeated effects such as tears and Z's
 *   - A small state machine for the different moods
 *
 * Serial control (115200 baud)
 * ----------------------------
 *   n = NORMAL
 *   h = HAPPY
 *   a = ANGRY
 *   t = TIRED
 *   s = SLEEPY
 *   l = LOVE
 *   d = SAD
 *   u = SURPRISED
 *
 * In AUTO_CYCLE mode, the character automatically changes mood every
 * MOOD_TIME_MS milliseconds unless a serial command or touch input temporarily
 * holds a mood.
 * ==========================================================================
 */

// ============================================================================
// 1. LIBRARIES
// ============================================================================

#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>


// ============================================================================
// 2. HARDWARE CONFIGURATION
// ============================================================================

#define SCREEN_W 128
#define SCREEN_H 64

#define OLED_ADDR 0x3C       // Use 0x3D if your OLED uses the other I2C address.
#define SDA_PIN   21
#define SCL_PIN   22

// Optional capacitive-touch input.
#define USE_TOUCH       0    // 1 = enable touch input, 0 = disable it
#define TOUCH_PIN       4
#define TOUCH_THRESHOLD 30

// Automatically cycle through all moods.
#define AUTO_CYCLE   1
#define MOOD_TIME_MS 5000

Adafruit_SSD1306 display(SCREEN_W, SCREEN_H, &Wire, -1);


// ============================================================================
// 3. MOOD DEFINITIONS
// ============================================================================

// Every mood is a state of the character. The drawing and animation behavior
// changes depending on the currently selected state.
enum Mood {
  NORMAL,
  HAPPY,
  ANGRY,
  TIRED,
  SLEEPY,
  LOVE,
  SAD,
  SURPRISED,
  MOOD_COUNT
};

// Human-readable names used by the Serial Monitor.
const char* moodNames[MOOD_COUNT] = {
  "NORMAL",
  "HAPPY",
  "ANGRY",
  "TIRED",
  "SLEEPY",
  "LOVE",
  "SAD",
  "SURPRISED"
};

Mood mood = NORMAL;


// ============================================================================
// 4. ANIMATION STATE
// ============================================================================

// --------------------------------------------------------------------------
// Eye position
// --------------------------------------------------------------------------
// curX / curY = where the eyes are currently located.
// tgtX / tgtY = where the eyes want to move.
//
// The animation system gradually moves curX/Y toward tgtX/Y instead of
// teleporting directly. This creates smooth eye movement.
float curX = 0;
float curY = 0;
float tgtX = 0;
float tgtY = 0;

// --------------------------------------------------------------------------
// Eye openness
// --------------------------------------------------------------------------
// openness = current amount by which the eye is open.
// tgtOpen  = desired amount of openness.
//
// 1.0 = fully open
// 0.0 = fully closed
float openness = 1.0;
float tgtOpen = 1.0;

// --------------------------------------------------------------------------
// Eye size
// --------------------------------------------------------------------------
// These are smoothly animated so that changing to SAD or SURPRISED does not
// instantly resize the eyes.
float curW = 32;
float curH = 40;

// --------------------------------------------------------------------------
// Mood transition pop
// --------------------------------------------------------------------------
// When a mood changes, pop starts at 1 and decays toward zero. It temporarily
// enlarges the eyes, creating a small "pop" effect during the transition.
float pop = 0;


// ============================================================================
// 5. TIMERS AND EVENT STATE
// ============================================================================

// Current time, updated once every loop iteration.
unsigned long nowMs = 0;

// Time at which the last frame was rendered. Used to limit the animation to
// roughly 30 frames per second.
unsigned long lastFrame = 0;

// Blink timing.
unsigned long nextBlink = 0;
unsigned long blinkEnd = 0;

// Looking-direction timing.
unsigned long nextLook = 0;

// Automatic mood-change timing.
unsigned long nextMood = 0;

// A temporary mood hold prevents AUTO_CYCLE from immediately replacing a
// mood selected by serial input or touch.
unsigned long holdUntil = 0;

// True while the character is currently closing its eyes for a blink.
bool blinking = false;


// ============================================================================
// 6. MOOD CONTROL
// ============================================================================

/*
 * Select a new mood and reset the animation events associated with a mood
 * change.
 *
 * The actual visual animation is handled later by animate() and drawEyes().
 * This function only establishes the new state and resets its timers.
 */
void setMood(Mood newMood) {
  mood = newMood;

  // Start the short eye-size "pop" transition.
  pop = 1.0;

  // Make sure we do not enter a new mood while a previous blink is active.
  blinking = false;

  // Give the new mood a fresh blink and look schedule.
  nextBlink = nowMs + 1500;
  nextLook = nowMs;

  Serial.print("Mood: ");
  Serial.println(moodNames[mood]);
}


// ============================================================================
// 7. SERIAL INPUT
// ============================================================================

/*
 * Read mood commands from the Serial Monitor.
 *
 * Only one character is needed for each mood, making manual testing quick.
 * Selecting a mood through Serial also holds that mood for 15 seconds so
 * AUTO_CYCLE does not immediately overwrite the user's choice.
 */
void handleSerial() {
  while (Serial.available()) {
    char c = Serial.read();
    int selectedMood = -1;

    if      (c == 'n') selectedMood = NORMAL;
    else if (c == 'h') selectedMood = HAPPY;
    else if (c == 'a') selectedMood = ANGRY;
    else if (c == 't') selectedMood = TIRED;
    else if (c == 's') selectedMood = SLEEPY;
    else if (c == 'l') selectedMood = LOVE;
    else if (c == 'd') selectedMood = SAD;
    else if (c == 'u') selectedMood = SURPRISED;

    if (selectedMood >= 0) {
      setMood((Mood)selectedMood);

      // Keep the manually selected mood active for 15 seconds.
      holdUntil = nowMs + 15000;
      nextMood = holdUntil;
    }
  }
}


// ============================================================================
// 8. BEHAVIOR / DECISION LOGIC
// ============================================================================

/*
 * Update the character's high-level behavior.
 *
 * This function does NOT draw anything.
 * It decides things such as:
 *   - Should the mood change?
 *   - Should the character blink?
 *   - Where should the eyes look next?
 *   - How open should the eyes be?
 */
void updateBehavior() {

  // --------------------------------------------------------------------------
  // Optional touch input
  // --------------------------------------------------------------------------
#if USE_TOUCH
  if (touchRead(TOUCH_PIN) < TOUCH_THRESHOLD) {
    if (mood != LOVE) {
      setMood(LOVE);
    }

    // Hold LOVE for 3 seconds after touch.
    holdUntil = nowMs + 3000;
    nextMood = holdUntil;
  }
#endif

  // --------------------------------------------------------------------------
  // Automatic mood cycling
  // --------------------------------------------------------------------------
#if AUTO_CYCLE
  if (nowMs >= holdUntil && nowMs >= nextMood) {
    // Move to the next mood and wrap back to NORMAL after SURPRISED.
    setMood((Mood)((mood + 1) % MOOD_COUNT));
    nextMood = nowMs + MOOD_TIME_MS;
  }
#endif

  // --------------------------------------------------------------------------
  // Blinking
  // --------------------------------------------------------------------------
  // Some moods look better without blinking because their special eye shape
  // already communicates the expression.
  bool canBlink = (
    mood == NORMAL ||
    mood == ANGRY ||
    mood == TIRED ||
    mood == SAD ||
    mood == SURPRISED
  );

  // Start a blink when its timer expires.
  if (!blinking && canBlink && nowMs >= nextBlink) {
    blinking = true;

    // Tired blinking is deliberately slower/heavier than a normal blink.
    blinkEnd = nowMs + (mood == TIRED ? 350 : 130);
  }

  // Finish the current blink and schedule the next one at a random time.
  if (blinking && nowMs >= blinkEnd) {
    blinking = false;
    nextBlink = nowMs + random(1500, 4500);
  }

  // TIRED eyes are naturally only 65% open even when they are not blinking.
  float baseOpen = (mood == TIRED) ? 0.65 : 1.0;

  // A blink temporarily overrides the normal eye openness.
  tgtOpen = blinking ? 0.0 : baseOpen;

  // --------------------------------------------------------------------------
  // Looking direction
  // --------------------------------------------------------------------------
  // The target position is changed occasionally. animate() then moves the
  // actual eye position smoothly toward this target.
  if (nowMs >= nextLook) {
    switch (mood) {

      case NORMAL:
        // Most of the time look somewhere random, but occasionally return to
        // the center. This prevents the character from constantly looking
        // around without pauses.
        if (random(3) == 0) {
          tgtX = 0;
          tgtY = 0;
        } else {
          tgtX = random(-10, 11);
          tgtY = random(-6, 7);
        }

        nextLook = nowMs + random(1000, 2500);
        break;

      case SAD:
        // Sad eyes mostly look downward, with a small random horizontal
        // variation so they do not appear completely static.
        tgtX = random(-4, 5);
        tgtY = 8;
        nextLook = nowMs + random(1500, 3000);
        break;

      case TIRED:
        // Tired eyes move less frequently and tend to look downward.
        tgtX = random(-6, 7);
        tgtY = 4;
        nextLook = nowMs + random(2000, 4000);
        break;

      default:
        // The remaining moods keep their eyes centered unless their special
        // animation modifies the drawing later.
        tgtX = 0;
        tgtY = 0;
        nextLook = nowMs + 1000;
        break;
    }
  }
}


// ============================================================================
// 9. LOW-LEVEL ANIMATION
// ============================================================================

/*
 * Smoothly update all continuously changing animation values.
 *
 * The key formula used here is:
 *
 *     current += (target - current) * speed;
 *
 * This is exponential smoothing. Instead of jumping directly to the target,
 * the value moves a fraction of the remaining distance every frame.
 *
 * Example:
 *     current = 0, target = 10, speed = 0.3
 *
 *     frame 1 -> 3.0
 *     frame 2 -> 5.1
 *     frame 3 -> 6.57
 *     ...
 *
 * This produces smooth motion without needing a complicated easing function.
 */
void animate() {

  // Desired eye dimensions for each mood.
  float targetW = 32;
  float targetH = 40;

  if (mood == SAD) {
    targetW = 30;
    targetH = 32;
  }

  if (mood == SURPRISED) {
    targetW = 36;
    targetH = 50;
  }

  // Smoothly resize the eyes instead of changing their dimensions instantly.
  curW += (targetW - curW) * 0.25;
  curH += (targetH - curH) * 0.25;

  // Smoothly move the eyes toward their target looking direction.
  curX += (tgtX - curX) * 0.3;
  curY += (tgtY - curY) * 0.3;

  // Tired eyes open/close more slowly than other moods.
  float opennessSpeed = (mood == TIRED) ? 0.15 : 0.5;
  openness += (tgtOpen - openness) * opennessSpeed;

  // Exponential decay of the mood-change pop effect.
  // Repeatedly multiplying by 0.85 makes the effect quickly approach zero.
  pop *= 0.85;
}


// ============================================================================
// 10. SHAPE DRAWING HELPERS
// ============================================================================

/*
 * Draw a heart using simple geometric primitives.
 *
 * Two circles form the upper lobes and a triangle forms the lower point.
 * The size parameter controls the overall scale of the heart.
 */
void drawHeart(int cx, int cy, int s) {
  display.fillCircle(cx - s / 2, cy - s / 4, s / 2 + 1, WHITE);
  display.fillCircle(cx + s / 2, cy - s / 4, s / 2 + 1, WHITE);
  display.fillTriangle(
    cx - s - 1, cy - s / 6,
    cx + s + 1, cy - s / 6,
    cx, cy + s,
    WHITE
  );
}


// ============================================================================
// 11. EYE RENDERING
// ============================================================================

/*
 * Draw one eye.
 *
 * cx, cy  = eye center
 * w, h    = eye dimensions
 * isLeft  = tells the function which side it is drawing so that asymmetric
 *           expressions such as ANGRY can be mirrored correctly.
 *
 * Most moods begin with the same rounded rectangle. Special moods then
 * modify that base shape using masks or additional shapes.
 */
void drawEye(int cx, int cy, int w, int h, bool isLeft) {
  int x = cx - w / 2;
  int y = cy - h / 2;

  // --------------------------------------------------------------------------
  // HAPPY: draw an arch instead of a normal eye.
  // --------------------------------------------------------------------------
  if (mood == HAPPY) {
    // Several circles create a thicker arch. The rectangle below masks the
    // lower half, leaving a shape similar to ^.
    for (int i = 0; i < 4; i++) {
      display.drawCircle(cx, cy + 8, 13 + i, WHITE);
    }

    display.fillRect(cx - 20, cy + 8, 41, 30, BLACK);
    return;
  }

  // --------------------------------------------------------------------------
  // LOVE: replace the eye completely with a pulsing heart.
  // --------------------------------------------------------------------------
  if (mood == LOVE) {
    // sin() makes the heart grow and shrink continuously, producing a
    // heartbeat-like pulse.
    int heartSize = 10 + (int)(3 * sin(nowMs * 0.01));
    drawHeart(cx, cy, heartSize);
    return;
  }

  // --------------------------------------------------------------------------
  // Base eye shape used by most moods.
  // --------------------------------------------------------------------------
  int radius = min(10, h / 2);
  display.fillRoundRect(x, y, w, h, radius, WHITE);

  // If the eye is extremely thin (for example during a blink), there is not
  // enough height for pupils or eyelid details.
  if (h < 10) {
    return;
  }

  // --------------------------------------------------------------------------
  // Mood-specific eyelids / pupils.
  // --------------------------------------------------------------------------
  int drop;

  switch (mood) {

    case ANGRY:
      // Remove a triangular section from the top of each eye to create the
      // characteristic inward/outward slanted angry eyelid.
      drop = h * 4 / 10;

      if (isLeft) {
        display.fillTriangle(
          x - 1, y - 1,
          x + w + 1, y - 1,
          x + w + 1, y + drop,
          BLACK
        );
      } else {
        display.fillTriangle(
          x - 1, y - 1,
          x + w + 1, y - 1,
          x - 1, y + drop,
          BLACK
        );
      }
      break;

    case TIRED:
    case SAD:
      // Remove part of the top of the eye to create a drooping eyelid.
      drop = (mood == SAD) ? h * 5 / 10 : h * 35 / 100;

      if (isLeft) {
        display.fillTriangle(
          x - 1, y - 1,
          x + w + 1, y - 1,
          x - 1, y + drop,
          BLACK
        );
      } else {
        display.fillTriangle(
          x - 1, y - 1,
          x + w + 1, y - 1,
          x + w + 1, y + drop,
          BLACK
        );
      }
      break;

    case SURPRISED:
      // The eye itself becomes larger. A small black circle acts as the
      // pupil and follows the current looking direction.
      if (h >= 20) {
        display.fillCircle(
          cx + (int)(curX / 2),
          cy + (int)(curY / 2),
          6,
          BLACK
        );
      }
      break;

    default:
      // NORMAL and SLEEPY need no additional eye geometry here.
      break;
  }

  // --------------------------------------------------------------------------
  // SAD: falling tear animation.
  // --------------------------------------------------------------------------
  if (mood == SAD) {
    // phase repeatedly goes from 0 to 99. Dividing time by 25 controls the
    // speed of the tear. The right tear starts halfway through the cycle so
    // the two tears do not fall simultaneously.
    int phase = (nowMs / 25 + (isLeft ? 0 : 50)) % 100;

    // Convert the phase into vertical movement below the eye.
    int tearY = y + h + phase * 16 / 100;

    // Stop drawing the tear near the end of the cycle and prevent it from
    // being drawn outside the visible OLED area.
    if (phase < 90 && tearY < 62) {
      display.fillCircle(cx, tearY, 2, WHITE);
    }
  }
}


// ============================================================================
// 12. SLEEPY MOOD EFFECTS
// ============================================================================

/*
 * Draw the three floating Z characters used by SLEEPY.
 *
 * All three Z's use the same movement equation, but each has a different
 * phase offset. This creates a stream of Z's without needing three separate
 * animation systems.
 */
void drawSleepZ() {
  display.setTextColor(WHITE);

  for (int i = 0; i < 3; i++) {
    // Each Z starts at a different point in the 0..99 animation cycle.
    int phase = (nowMs / 40 + i * 33) % 100;

    // Move diagonally upward as the phase increases.
    int zx = 92 + phase / 6 + i * 4;
    int zy = 28 - phase * 28 / 100;

    // Make the last Z larger so the floating effect has some visual variety.
    display.setTextSize(i == 2 ? 2 : 1);
    display.setCursor(zx, zy);
    display.print(i == 2 ? "Z" : "z");
  }
}


// ============================================================================
// 13. FRAME RENDERING
// ============================================================================

/*
 * Build one complete OLED frame.
 *
 * This function is responsible only for turning the current animation state
 * into pixels. The decision-making happens in updateBehavior(), while smooth
 * numerical animation happens in animate().
 */
void drawEyes() {
  display.clearDisplay();

  // --------------------------------------------------------------------------
  // Apply the temporary mood-change "pop" to the eye dimensions.
  // --------------------------------------------------------------------------
  int w = (int)(curW * (1 + 0.25 * pop));
  int hBase = (int)(curH * (1 + 0.25 * pop));

  // Scale the eye height according to openness.
  int h = max(3, (int)(hBase * openness));

  // Base vertical position plus the current looking offset.
  int cy = 32 + (int)curY;

  // Current horizontal looking offset.
  int dx = (int)curX;

  // --------------------------------------------------------------------------
  // Mood-specific whole-eye motion.
  // --------------------------------------------------------------------------
  switch (mood) {

    case HAPPY:
      // Sine wave creates smooth up/down bouncing.
      cy += (int)(sin(nowMs * 0.012) * 3);
      break;

    case ANGRY:
      // Small random displacement creates an intentionally rough shake.
      dx += random(-2, 3);
      break;

    case SLEEPY:
      // Slow sine wave changes eye height, creating a breathing-like motion.
      h = 3 + (int)(2.5 + 2.5 * sin(nowMs * 0.003));
      cy = 40;
      break;

    default:
      break;
  }

  // Draw the left and right eyes symmetrically around the center of the OLED.
  drawEye(64 - 28 + dx, cy, w, h, true);
  drawEye(64 + 28 + dx, cy, w, h, false);

  // SLEEPY has an additional floating-Z effect.
  if (mood == SLEEPY) {
    drawSleepZ();
  }

  // Transfer the completed frame from the SSD1306 buffer to the physical OLED.
  display.display();
}


// ============================================================================
// 14. SETUP
// ============================================================================

void setup() {
  // Start Serial communication for debugging and manual mood selection.
  Serial.begin(115200);

  // Initialize I2C using the selected SDA and SCL pins.
  Wire.begin(SDA_PIN, SCL_PIN);

  // Initialize the SSD1306 OLED.
  if (!display.begin(SSD1306_SWITCHCAPVCC, OLED_ADDR)) {
    Serial.println("SSD1306 not found - check wiring/address");

    // Stop here if the display cannot be initialized.
    for (;;) {
      delay(1000);
    }
  }

  // Seed the random-number generator so random looking directions, blink
  // intervals, and angry shaking are different after each reset.
  randomSeed(esp_random());

  // Clear any old pixels from the display.
  display.clearDisplay();
  display.display();

  // Initialize all time-based animation events.
  nowMs = millis();
  nextMood = nowMs + MOOD_TIME_MS;
  nextBlink = nowMs + 2000;

  Serial.println("Dasai Mochi started");
}


// ============================================================================
// 15. MAIN LOOP
// ============================================================================

void loop() {
  // Always keep the software clock updated.
  nowMs = millis();

  // Check for user commands before updating the animation.
  handleSerial();

  // Render at approximately 30 FPS.
  // 1000 ms / 33 ms ≈ 30.3 frames per second.
  if (nowMs - lastFrame < 33) {
    return;
  }

  lastFrame = nowMs;

  // The animation pipeline is deliberately kept in this order:
  //
  //   behavior -> numerical animation -> drawing
  //
  // This makes it easy to understand and extend the character later.
  updateBehavior();
  animate();
  drawEyes();
}

# Final review — what you should now recognize

The complete program looks large, but almost all of its animation behaviour comes from a few reusable ideas.

## 1. Interpolation

```cpp
current += (target - current) * speed;
```

Used for:

- looking
- blinking
- resizing
- transitions

## 2. Sine waves

```cpp
sin(time * frequency)
```

Used for:

- HAPPY bounce
- SLEEPY breathing
- LOVE pulse

## 3. Randomness

```cpp
random(min, max)
```

Used for:

- looking direction
- blink interval
- ANGRY jitter

## 4. Phase

```cpp
(time + offset) % period
```

Used for:

- tears
- Z's
- repeated effects

## 5. State machine

```text
NORMAL
HAPPY
ANGRY
TIRED
SLEEPY
LOVE
SAD
SURPRISED
```

The state chooses the visual rules.

## 6. Procedural rendering

The character is generated from:

```text
rounded rectangles
circles
triangles
text
masks
```

rather than stored animation frames.

### The central lesson

\[
\boxed{
\text{Emotion}
=
\text{Shape}
+
\text{Movement}
+
\text{Timing}
+
\text{Interaction}
}
\]

and:

\[
\boxed{
\text{Animation}
=
\text{Interpolation}
+
\text{Sine Waves}
+
\text{Randomness}
+
\text{Phase}
}
\]

That is the mathematical core of the Mochi animation engine.
## Interpreting values as pixels

All coordinates and dimensions in the sketches are OLED pixels, not percentages or millimetres. On a 128 × 64 matrix, one x-pixel is 1/128 of the width and one y-pixel is 1/64 of the height. Because the matrix is 2:1, x and y offsets of the same pixel value represent different fractions of the screen.
